# HY3D Worker Colab

Worker manual y worker por Google Drive para HY3D v2.

Este notebook no acepta, no edita y no exporta STL.
Su única responsabilidad es recibir `job_package.zip`, ejecutar TripoSR Clean y devolver `result_package.zip`.

In [ ]:
from __future__ import annotations

import json
import os
import shutil
import subprocess
import traceback
import zipfile
from datetime import datetime, timezone
from pathlib import Path

from PIL import Image

try:
    from google.colab import drive, files  # type: ignore
    IN_COLAB = True
except Exception:
    drive = None
    files = None
    IN_COLAB = False

WORKDIR = Path('/content/hy3d_v2_worker')
MANUAL_DIR = WORKDIR / 'manual'
TRIPOSR_DIR = Path('/content/TripoSR')
CLOUD_ROOT = Path('/content/drive/MyDrive/HY3D_V2_CLOUD')
INCOMING_DIR = CLOUD_ROOT / 'incoming'
PROCESSING_DIR = CLOUD_ROOT / 'processing'
COMPLETED_DIR = CLOUD_ROOT / 'completed'
FAILED_DIR = CLOUD_ROOT / 'failed'
LOGS_DIR = CLOUD_ROOT / 'logs'
NOTEBOOKS_DIR = CLOUD_ROOT / 'notebooks'
PROCESS_ONLY_ONE_JOB = True

for path in (WORKDIR, MANUAL_DIR):
    path.mkdir(parents=True, exist_ok=True)

def utc_now_iso() -> str:
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat()

def clean_dir(path: Path) -> None:
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)

def ensure_drive_dirs() -> None:
    for path in (CLOUD_ROOT, INCOMING_DIR, PROCESSING_DIR, COMPLETED_DIR, FAILED_DIR, LOGS_DIR, NOTEBOOKS_DIR):
        path.mkdir(parents=True, exist_ok=True)

def safe_extract_zip(archive_path: Path, destination: Path) -> list[Path]:
    extracted: list[Path] = []
    with zipfile.ZipFile(archive_path, 'r') as archive:
        for info in archive.infolist():
            member = Path(info.filename)
            if info.is_dir():
                continue
            if member.is_absolute() or '..' in member.parts or ':' in info.filename.replace('\\', '/'):
                raise RuntimeError(f'Unsafe ZIP entry rejected: {info.filename}')
            target = destination / member
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(info, 'r') as src, open(target, 'wb') as dst:
                shutil.copyfileobj(src, dst)
            extracted.append(target)
    return extracted

def run_command(command: list[str], cwd: Path | None = None, log_path: Path | None = None) -> subprocess.CompletedProcess[str]:
    prefix = f"$ {' '.join(command)}\n"
    if log_path is not None:
        log_path.parent.mkdir(parents=True, exist_ok=True)
        with open(log_path, 'a', encoding='utf-8') as log_file:
            log_file.write(prefix)
            result = subprocess.run(command, cwd=str(cwd) if cwd else None, capture_output=True, text=True)
            log_file.write(result.stdout)
            if result.stderr:
                log_file.write('\n[stderr]\n')
                log_file.write(result.stderr)
            log_file.write(f'\n[returncode] {result.returncode}\n\n')
    else:
        result = subprocess.run(command, cwd=str(cwd) if cwd else None, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"Command failed ({result.returncode}): {' '.join(command)}\n{result.stderr}")
    return result

def install_triposr(log_path: Path) -> None:
    if not TRIPOSR_DIR.exists():
        run_command(['git', 'clone', 'https://github.com/VAST-AI-Research/TripoSR.git', str(TRIPOSR_DIR)], log_path=log_path)
    run_command(['python', '-m', 'pip', 'install', '--upgrade', 'pip', 'setuptools', 'wheel'], log_path=log_path)
    run_command(['python', '-m', 'pip', 'install', 'numpy==1.26.4'], log_path=log_path)
    run_command(['python', '-m', 'pip', 'install', '-r', str(TRIPOSR_DIR / 'requirements.txt')], log_path=log_path)
    try:
        run_command(['python', '-m', 'pip', 'install', 'git+https://github.com/tatsy/torchmcubes.git'], log_path=log_path)
    except Exception as exc:
        with open(log_path, 'a', encoding='utf-8') as log_file:
            log_file.write(f'[warning] torchmcubes reinstall skipped: {exc}\n')

def find_primary_image(extract_dir: Path) -> Path:
    preferred = extract_dir / 'input' / 'primary_image.png'
    if preferred.exists():
        return preferred
    for suffix in ('.png', '.jpg', '.jpeg', '.webp', '.avif', '.bmp'):
        matches = list((extract_dir / 'input').glob(f'primary_image*{suffix}'))
        if matches:
            return matches[0]
    raise FileNotFoundError('input/primary_image.png was not found in job_package.zip')

def normalize_primary_image(source: Path, destination: Path) -> Path:
    destination.parent.mkdir(parents=True, exist_ok=True)
    with Image.open(source) as image:
        image.convert('RGB').save(destination, format='PNG')
    return destination

def package_result(job_id: str, version_id: str, generated_glb: Path, engine_log_path: Path, output_dir: Path, result_zip_path: Path) -> dict:
    engine_output_dir = output_dir / 'engine_output'
    logs_dir = output_dir / 'logs'
    engine_output_dir.mkdir(parents=True, exist_ok=True)
    logs_dir.mkdir(parents=True, exist_ok=True)
    root_glb = output_dir / 'model.glb'
    engine_glb = engine_output_dir / 'model.glb'
    shutil.copy2(generated_glb, root_glb)
    shutil.copy2(generated_glb, engine_glb)
    local_engine_log = logs_dir / 'engine_log.txt'
    shutil.copy2(engine_log_path, local_engine_log)
    result_manifest = {
        'result_package_version': 1,
        'job_id': job_id,
        'version_id': version_id,
        'engine_id': 'triposr_clean',
        'candidate_path': 'engine_output/model.glb',
        'generated_at': utc_now_iso(),
        'status': 'completed',
    }
    result_manifest_path = output_dir / 'result_manifest.json'
    result_manifest_path.write_text(json.dumps(result_manifest, indent=2) + '\n', encoding='utf-8')
    if result_zip_path.exists():
        result_zip_path.unlink()
    with zipfile.ZipFile(result_zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
        archive.write(result_manifest_path, 'result_manifest.json')
        archive.write(root_glb, 'model.glb')
        archive.write(engine_glb, 'engine_output/model.glb')
        archive.write(local_engine_log, 'logs/engine_log.txt')
    return result_manifest

def run_triposr_for_job(job_package_path: Path, workspace: Path, engine_log_path: Path) -> tuple[dict, Path]:
    extract_dir = workspace / 'job_package'
    output_dir = workspace / 'output'
    triposr_output_dir = workspace / 'triposr_run_output'
    clean_dir(extract_dir)
    clean_dir(output_dir)
    clean_dir(triposr_output_dir)
    extracted_files = safe_extract_zip(job_package_path, extract_dir)
    job_manifest_path = extract_dir / 'job_manifest.json'
    if not job_manifest_path.exists():
        raise RuntimeError('job_package.zip is missing job_manifest.json')
    job_manifest = json.loads(job_manifest_path.read_text(encoding='utf-8'))
    job_id = job_manifest.get('job_id')
    version_id = job_manifest.get('active_version', 'v1')
    if not job_id:
        raise RuntimeError('job_manifest.json is missing job_id')
    primary_image = find_primary_image(extract_dir)
    normalized_image = normalize_primary_image(primary_image, workspace / 'normalized_primary.png')
    with open(engine_log_path, 'a', encoding='utf-8') as log_file:
        log_file.write(f'[info] extracted_files={len(extracted_files)}\n')
        log_file.write(f'[info] primary_image={primary_image}\n')
    device = 'cuda' if os.path.exists('/usr/bin/nvidia-smi') else 'cpu'
    if device == 'cpu':
        with open(engine_log_path, 'a', encoding='utf-8') as log_file:
            log_file.write('[warning] GPU was not detected. TripoSR will run on CPU and may be too slow or fail due to limits.\n')
    command = [
        'python',
        str(TRIPOSR_DIR / 'run.py'),
        str(normalized_image),
        '--output-dir',
        str(triposr_output_dir),
        '--model-save-format',
        'glb',
        '--device',
        device,
    ]
    run_command(command, cwd=TRIPOSR_DIR, log_path=engine_log_path)
    glb_candidates = sorted(triposr_output_dir.rglob('*.glb'))
    if not glb_candidates:
        raise RuntimeError('TripoSR finished without producing model.glb. Check logs/engine_log.txt.')
    return {'job_id': job_id, 'version_id': version_id, 'device': device}, glb_candidates[0]

print(f'Workspace ready: {WORKDIR}')
print(f'Running inside Colab: {IN_COLAB}')

## Modo manual

Usa este flujo si quieres subir un `job_package.zip` directamente desde tu equipo y descargar `result_package.zip` al final.

In [ ]:
# Instalar TripoSR una sola vez por runtime
manual_engine_log = MANUAL_DIR / 'manual_engine_log.txt'
manual_engine_log.parent.mkdir(parents=True, exist_ok=True)
install_triposr(manual_engine_log)
print('TripoSR installation step completed.')

In [ ]:
# Subir job_package.zip, procesarlo y descargar result_package.zip
if not IN_COLAB:
    raise RuntimeError('This notebook is designed for Google Colab.')
clean_dir(MANUAL_DIR)
manual_engine_log = MANUAL_DIR / 'manual_engine_log.txt'
uploaded = files.upload()
if not uploaded:
    raise RuntimeError('No file uploaded. Please upload job_package.zip.')
uploaded_name = next(iter(uploaded))
job_package_path = MANUAL_DIR / 'job_package.zip'
shutil.move(str(MANUAL_DIR / uploaded_name), str(job_package_path))
job_context, generated_glb = run_triposr_for_job(job_package_path, MANUAL_DIR, manual_engine_log)
manual_result_zip = MANUAL_DIR / 'result_package.zip'
manual_result_manifest = package_result(job_context['job_id'], job_context['version_id'], generated_glb, manual_engine_log, MANUAL_DIR / 'result_package_root', manual_result_zip)
print(json.dumps(manual_result_manifest, indent=2))
files.download(str(manual_result_zip))

## Modo Google Drive Worker

Este modo monta Google Drive, procesa por defecto un solo ZIP desde `incoming/` y escribe el resultado en `completed/` o el fallo en `failed/`.

In [ ]:
# Montar Drive y preparar estructura cloud
if not IN_COLAB:
    raise RuntimeError('This notebook is designed for Google Colab.')
drive.mount('/content/drive')
ensure_drive_dirs()
print('CLOUD_ROOT =', CLOUD_ROOT)
print('PROCESS_ONLY_ONE_JOB =', PROCESS_ONLY_ONE_JOB)

In [ ]:
# Instalar TripoSR para el worker de Drive
drive_worker_log = LOGS_DIR / 'worker_setup.log'
install_triposr(drive_worker_log)
print('Drive worker setup completed.')

In [ ]:
# Procesar jobs en incoming/
ensure_drive_dirs()
incoming_jobs = sorted(INCOMING_DIR.glob('*.zip'))
if PROCESS_ONLY_ONE_JOB and incoming_jobs:
    incoming_jobs = incoming_jobs[:1]
if not incoming_jobs:
    print('No jobs found in incoming/.')
else:
    for incoming_zip in incoming_jobs:
        started_at = utc_now_iso()
        processing_zip = PROCESSING_DIR / incoming_zip.name
        if processing_zip.exists():
            processing_zip.unlink()
        shutil.move(str(incoming_zip), str(processing_zip))
        job_id = processing_zip.name.replace('_job_package.zip', '')
        status_json_name = f'{job_id}_status.json'
        processing_status_path = PROCESSING_DIR / status_json_name
        engine_log_path = LOGS_DIR / f'{job_id}_engine_log.txt'
        processing_status_path.write_text(json.dumps({'job_id': job_id, 'engine_id': 'triposr_clean', 'status': 'processing', 'started_at': started_at}, indent=2) + '\n', encoding='utf-8')
        try:
            job_workspace = WORKDIR / 'drive_worker' / job_id
            clean_dir(job_workspace)
            job_context, generated_glb = run_triposr_for_job(processing_zip, job_workspace, engine_log_path)
            result_zip_path = COMPLETED_DIR / f"{job_context['job_id']}_result_package.zip"
            package_result(job_context['job_id'], job_context['version_id'], generated_glb, engine_log_path, job_workspace / 'result_package_root', result_zip_path)
            completed_status = {
                'job_id': job_context['job_id'],
                'version_id': job_context['version_id'],
                'engine_id': 'triposr_clean',
                'status': 'completed',
                'result_package': result_zip_path.name,
                'started_at': started_at,
                'finished_at': utc_now_iso(),
                'error': None,
            }
            (COMPLETED_DIR / status_json_name).write_text(json.dumps(completed_status, indent=2) + '\n', encoding='utf-8')
            processing_status_path.unlink(missing_ok=True)
            print(f"Completed job: {job_context['job_id']} -> {result_zip_path}")
        except Exception as exc:
            failed_error = {
                'job_id': job_id,
                'engine_id': 'triposr_clean',
                'status': 'failed',
                'error': str(exc),
                'traceback': traceback.format_exc(),
                'failed_at': utc_now_iso(),
            }
            (FAILED_DIR / f'{job_id}_error.json').write_text(json.dumps(failed_error, indent=2) + '\n', encoding='utf-8')
            if engine_log_path.exists():
                shutil.copy2(engine_log_path, FAILED_DIR / f'{job_id}_engine_log.txt')
            shutil.copy2(processing_zip, FAILED_DIR / processing_zip.name)
            processing_status_path.unlink(missing_ok=True)
            print(f'Failed job: {job_id} -> {exc}')